# 7단계 로컬 운영 진입점
배치 → 저장 모델 추론 → 분포 변화 감지 → 원인 확인 후 후보 생성 → 분리 평가 → 교체/롤백.
기본 실행은 상태 조회만 합니다. 독립 평가 자료와 현장 기준이 없는 자동 운영은 시작하지 않습니다.
CLI의 각 명령과 같은 공통 구현을 사용합니다. 자세한 예시는 common/OPERATIONS_GUIDE.txt를 참고하세요.

In [1]:
from pathlib import Path
import sys
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'03.modeling').is_dir())
sys.path.insert(0,str(ROOT/'03.modeling/common'))
from decision_runtime import Operations
DATASET='rg3'
ops=Operations(DATASET)
ops.registry()

{'active_ocsvm': None,
 'fixed_if': 'if-7e69cab8fdd04a17',
 'history': [{'action': 'fix_if_reference',
   'version': 'if-7e69cab8fdd04a17',
   'at': '2026-10-02T01:43:12.412776+00:00'}],
 'active_models': {'if': 'if-7e69cab8fdd04a17'}}

## 명시적으로 수행할 작업
1. 초기 후보의 버전과 지정 사유를 확인한 후 `ops.initialize(version, reason)`.
2. `ops.ingest(frame, label_source="실제 출처", coordinates_confirmed=True)`.
3. 표본이 누적되면 드리프트가 계산됩니다. 생산 시간 미확인 자료로 시간 성능을 주장하지 않습니다.
4. 원인 확인 후 `ops.retrain(kind, batch_ids, drift_id, cause)`.
5. `ops.register_evaluation(frame, label_source)`로 학습과 별개인 평가 패턴을 예약.
6. `ops.evaluate(candidate, evaluation_id)` 결과와 분포 안내 확인.
7. 통과한 평가만 `ops.promote(assessment_id)`. 필요시 `ops.rollback(kind, reason)`.
비라벨은 추론·감지만 가능하고 재학습에는 쓰지 않습니다.
RG3의 구간 안내에는 참조·평가 관측 수, 위험 이력, Wilson 구간과 표본 부족 표시가 들어갑니다.

### 최근 배치 판단 결과 (읽기 전용)
CN7은 RF 검사 우선순위(상위 검사 비율), RG3는 판단 불확실 단계(높음·중간·낮음)를 보여줍니다. 불량 확률이 아닙니다.

In [ ]:
# 읽기 전용: 최근 배치의 판단 결과 확인 (변경 작업 없음)
import json
ledger=ops.state/'batch_ledger.json'
ids=json.loads(ledger.read_text(encoding='utf-8'))['batches'] if ledger.exists() else []
if not ids:
    print('수집된 배치 없음')
else:
    folder=ops.state/'batches'/ids[-1]
    for name in ['inspection_priority.txt','inspection_advice.txt']:   # CN7 검사 우선순위 / RG3 불확실 단계
        if (folder/name).exists():
            print(f'[{ids[-1]}] {name}');print('\n'.join((folder/name).read_text(encoding='utf-8').splitlines()[:25]))